<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_03_compare_and_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 · Notebook 03 — Compare, and Report

**Paired with L7.1 · Fundamentals of PINNs**

Two solutions to the same slot, one with the walls as a penalty and one with
them built in. This notebook puts the numbers side by side, asks one question
neither notebook could answer alone, and assembles the report.

What is marked is not whether your numbers match anyone else's. It is whether
you can say what you measured, what it means, and where it stops being true.

---

## 0 · Setup and what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
soft = np.load(os.path.join("Ex07.1_outputs", "nb01_soft.npz"))
hard = np.load(os.path.join("Ex07.1_outputs", "nb02_hard.npz"))

X, Y, pts = grid_points(161, 161, pb.DOMAIN)
theta_ref = pb.theta_exact(pts[:, 0], pts[:, 1])

print(error_table(
    [["soft, w = 1", f"{float(soft['rel']):.3e}",
      f"{float(soft['max_err']):.4f}", f"{float(soft['wall']):.2e}"],
     ["hard", f"{float(hard['rel']):.3e}",
      f"{float(hard['max_err']):.4f}", f"{float(hard['wall']):.2e}"]],
    ["enforcement", "relative L2", "worst error [K]", "worst wall [K]"]))

## 0b · Your personal seed

Every notebook in this set fixes the seed to 88 so the printed "what you should
see" blocks are true on any machine. That is right for checking your work and
wrong for reporting it — with one seed the whole cohort produces identical
numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints where the questions ask for it. Your supervisor can
regenerate exactly these numbers from your study number alone.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own collocation draw, and what the manufactured source looks like on it.
your_pts = interior_points(2000, pb.DOMAIN, method="lhs", seed=SEED)
q_you = pb.source(your_pts[:, 0], your_pts[:, 1])
theta_you = pb.theta_exact(your_pts[:, 0], your_pts[:, 1])

print()
print(f"  your mean source        : {q_you.mean()/1e6:.5f} MW/m^3")
print(f"  your mean excess temp   : {theta_you.mean():.5f} K")
print(f"  your implied J          : {pb.equivalent_current_density(q_you.mean()):.4f} A/mm^2")

## 1 · The question neither notebook asked

Both models were trained on 2000 collocation points. Does either method degrade
faster than the other when points are scarce? This matters: in three dimensions
the same density costs cubically more, and Ex_09 and Ex_10 both run at
densities you would not choose.

### Your turn

In [ ]:
# TODO: retrain both methods at several collocation counts.
#
#   COUNTS = [200, 500, 1000, 2000, 4000]
#
#   For each n, and for each of the two methods:
#       set_seed(88); MLP(n_in=2, n_hidden=32, n_layers=4)
#       xy_f = to_tensor(interior_points(n, pb.DOMAIN, seed=1), requires_grad=True)
#       soft also needs xy_b = to_tensor(boundary_points(80, pb.DOMAIN, seed=1))
#       train_two_stage(..., adam_steps=1500, lbfgs_steps=80, report_every=0)
#       score relative_l2 on the 161x161 grid
#
#   Put them in scarcity = {"soft": [...], "hard": [...]}, aligned with COUNTS.
#
#   You will need pde_residual, make_loss, theta_trial, pde_residual_hard and
#   make_loss_hard from notebooks 01 and 02. Copy them in -- retyping them is
#   how you find out whether you understood them.
#
# Ten training runs; a few minutes.

raise NotImplementedError("Sweep the collocation count for both methods")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for i, (name, curve) in enumerate(scarcity.items()):
    ax.plot(COUNTS, curve, "o-", lw=1.9, ms=6, color=CYCLE[i], label=name)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("collocation points"); ax.set_ylabel("relative L2")
ax.set_title("Which method minds a sparse sample more?")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

print(error_table(
    [[str(n), f"{s:.3e}", f"{h:.3e}", f"{s/h:.2f}x"]
     for n, s, h in zip(COUNTS, scarcity["soft"], scarcity["hard"])],
    ["points", "soft", "hard", "soft/hard"]))

**What you should see.** Both curves falling as points are added, and the hard
curve below the soft one at every count — but look at the *ratio* column rather
than the absolute values. If the ratio grows as points get scarcer, hard
enforcement is not merely better, it is disproportionately better where it is
hardest to sample. Say which you observed.

There is a reason to expect that. With few points, the soft model has to spend
some of its limited information learning something it was *told* — that the
walls are at zero. The hard model is given that for free and can spend
everything on the physics.

---

## 2 · Your answers

Replace every string. Keep to the word limits; they are tight on purpose.

In [ ]:
# TODO: write your report. Every string below must be replaced.

Q1_WHY_LEAST_SQUARES = """
(120 words) The soft loss adds a PDE residual in W/m^3 to a wall residual in K.
Explain what dividing the physics residual by the peak source achieved, and
what w = 1 would have meant without it. Then state what the correct weight
would be if both residuals were measurements with known noise levels.
"""

Q2_THE_TRADE = """
(120 words) The soft model left a wall error of order 1e-2 K after training,
and it was not a failure to converge. Explain, in terms of the loss, why the
optimiser chose that -- and what it bought in exchange.
"""

Q3_HARD_LIMITS = """
(150 words) Hard enforcement removed a hyperparameter you could not have tuned
honestly. Name what it added in exchange, using your own experience of writing
the lift function g in notebook 02. Then say which of the two you would use for
a prescribed heat flux on one wall, and why the other does not apply.
"""

Q4_SCARCITY = """
(120 words) Report your soft/hard ratio at the smallest and largest collocation
counts. Say whether the gap widened or narrowed as points became scarce, and
give the mechanism you think is responsible. If your result contradicts the
prediction in section 1, say so -- that is a finding, not an error.
"""

Q5_HONEST_NUMBERS = """
(150 words) Notebook 00 printed an implied current density of 10.4 A/mm^2 and
confirmed the manufactured source was non-negative everywhere. Explain why both
checks belong in a manufactured problem. Then quote your personal-seed numbers
from section 0b and say what they would have looked like had the skew exceeded
1/3.
"""

Q6_WHAT_YOU_DISTRUST = """
(120 words) Name the result in this exercise you trust least, and say exactly
what experiment would settle it. An answer naming a specific number and a
specific test scores higher than a general statement about needing more data.
"""

NAME = "your name"
GROUP = "your group"

raise NotImplementedError("Write your report, then delete this line")

## 3 · Check, assemble, save

In [ ]:
answers = {
    "1 · Units in a composite loss": (Q1_WHY_LEAST_SQUARES, 120),
    "2 · What the optimiser traded": (Q2_THE_TRADE, 120),
    "3 · What hard enforcement costs": (Q3_HARD_LIMITS, 150),
    "4 · Behaviour under a sparse sample": (Q4_SCARCITY, 120),
    "5 · Keeping a manufactured problem honest": (Q5_HONEST_NUMBERS, 150),
    "6 · What you distrust": (Q6_WHAT_YOU_DISTRUST, 120),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or f"({limit} words)" in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    lines = ["# Ex_07.1 — Fundamentals of PINNs: a stator slot", "",
             f"**{NAME}** · {GROUP}", "",
             f"study number {STUDENT_NUMBER} · seed {SEED}", "",
             "Deep Learning for Engineering · Aalborg University · 2026", "",
             "---", ""]
    for title, (text, _) in answers.items():
        lines += [f"## {title}", "", text.strip(), ""]
    report = "\n".join(lines)
    out = os.path.join("Ex07.1_outputs", "Ex07.1_report.md")
    with open(out, "w", encoding="utf-8") as fh:
        fh.write(report)
    print("wrote", out)
    print(f"{sum(len(t.split()) for t, _ in answers.values())} words total")

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex07.1_report.md into Ex07.1_report.pdf, with any figure
# saved as Ex07.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open("Ex07.1_report.md", encoding="utf-8").read()
figs = sorted(glob.glob("Ex07.1_report*.png"))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf("Ex07.1_report.pdf")
print("written Ex07.1_report.pdf", f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download("Ex07.1_report.pdf")
except ImportError:
    pass


## 4 · What Ex_07.1 was for

One equation, solved twice, with the difference between the two runs being
where a piece of known information was put.

* **A boundary condition in the loss is a request.** The optimiser will trade
  it against everything else in the loss, and it will do so silently.
* **A boundary condition in the function space is a fact.** It holds before
  training starts and cannot be traded.
* **Prefer facts to requests, when you can construct them** — and know the
  three situations where you cannot: awkward geometry, non-zero boundary data,
  and conditions on derivatives.
* **Non-dimensionalise every residual before weighting it**, or the weights
  encode your unit system rather than your engineering.

That last point returns in every remaining exercise of Part 2, and L6.1 already
made it about $\lambda$ in a multi-term loss. It is the same argument.

Next: **Ex_07.2**, where time enters and the question becomes how many initial
conditions a problem actually needs.